# Causal trace

A causal trace runs a prompt clean and corrupted, then patches one site at one token position from the clean run into the corrupted run and measures how much of the clean output comes back (Meng, Bau, Andonian and Belinkov 2022, "Locating and Editing Factual Associations in GPT", NeurIPS 2022, arXiv:2202.05262). The score for each site and position is the drop in output divergence.

The model is a tiny one-block transformer built here, so the notebook runs offline in seconds. For another model, adapt the site selector, the token axis and the output metric.

In [ ]:
# recipe-test: tool=matplotlib expected=(1, 4)
from collections.abc import Callable
from typing import Any

import torch
from torch import nn

import torchlens as tl


class TinyTransformer(nn.Module):
    """Small deterministic transformer-like model for recipe notebooks."""

    def __init__(
        self,
        *,
        vocab_size: int = 11,
        d_model: int = 8,
        n_heads: int = 2,
        max_len: int = 6,
    ) -> None:
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(max_len, d_model))
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln_1 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.ReLU(),
            nn.Linear(d_model * 2, d_model),
        )
        self.ln_2 = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        positions = self.pos_embed[: tokens.shape[1]].unsqueeze(0)
        hidden = self.token_embed(tokens) + positions
        attended, _weights = self.attn(hidden, hidden, hidden, need_weights=False)
        hidden = self.ln_1(hidden + attended)
        hidden = self.ln_2(hidden + self.mlp(hidden))
        return self.head(hidden)


torch.manual_seed(63)
model = TinyTransformer().eval()
clean_tokens = torch.tensor([[1, 2, 3, 4]])
corrupt_tokens = torch.tensor([[1, 7, 3, 4]])
RESULT = tuple(clean_tokens.shape)

Trace the clean and the corrupted prompt. Every trace keeps the replay metadata the later patches need.

In [ ]:
# recipe-test: tool=matplotlib expected=True
clean_trace = tl.trace(model, clean_tokens)
corrupt_trace = tl.trace(model, corrupt_tokens)

clean_traceits = clean_trace.output_ops[-1].out.detach()
corrupt_traceits = corrupt_trace.output_ops[-1].out.detach()


def output_divergence(candidate_logits: torch.Tensor) -> float:
    """L2 distance between the last-position logits and the clean ones."""
    delta = candidate_logits[:, -1, :] - clean_traceits[:, -1, :]
    return float(torch.linalg.vector_norm(delta).detach())


corrupt_divergence = output_divergence(corrupt_traceits)
print(f"Corrupt output divergence: {corrupt_divergence:.4f}")
assert corrupt_divergence > 0.0
RESULT = corrupt_divergence > 0.0

`find_sites(tl.func("relu"))` lists every ReLU site; the sweep patches each one at each token position from the clean trace into the corrupted trace. This toy has one ReLU site, after its only attention layer, and every op downstream of it is token-local, so only the last position can move the last-token score; the earlier positions are zero-effect controls.

In [ ]:
# recipe-test: tool=matplotlib expected=True
def patch_position(source_out: torch.Tensor, position: int) -> Callable[..., torch.Tensor]:
    """Hook that replaces one sequence position with the clean activation."""

    def _hook(out: torch.Tensor, *, ctx: Any) -> torch.Tensor:
        patched = out.clone()
        patched[:, position, :] = source_out[:, position, :].to(
            device=out.device,
            dtype=out.dtype,
        )
        return patched

    return _hook


site_labels = sorted(set(corrupt_trace.find_sites(tl.func("relu"), max_fanout=20).labels()))
print(site_labels)
assert site_labels
RESULT = bool(site_labels)

In [ ]:
# recipe-test: tool=matplotlib expected=True
scores = torch.zeros(len(site_labels), clean_tokens.shape[1])
for site_index, label in enumerate(site_labels):
    clean_out = clean_trace.find_sites(tl.label(label)).first().out.detach()
    for position in range(clean_tokens.shape[1]):
        patched = corrupt_trace.fork(f"patch_{label}_pos_{position}")
        patched.do(tl.label(label), patch_position(clean_out, position))
        patched_logits = patched.output_ops[-1].out.detach()
        scores[site_index, position] = corrupt_divergence - output_divergence(patched_logits)

print(scores)
assert torch.isfinite(scores).all()
RESULT = bool(torch.isfinite(scores).all())

Positive scores mean the patch moved the corrupted output toward the clean one. `causal_trace_heatmap` takes any 2D score array.

In [ ]:
# recipe-test: tool=matplotlib expected=True
ax = tl.viz.causal_trace_heatmap(scores, cmap="viridis")
ax.set_xlabel("token position")
ax.set_ylabel("site")
ax.set_yticks(range(len(site_labels)), site_labels)
ax.set_title("Clean-patch rescue score")
RESULT = ax is not None